# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,LNG,49,1,0.998300,2026-05-14 11:08:46+00:00
1,RKLB,13,8,0.088438,2026-05-15 23:10:06+00:00
2,NBIS,9,8,0.421963,2026-05-15 23:10:06+00:00
3,APLD,33,1,0.999400,2026-05-15 15:10:18+00:00
4,NVDA,8,6,0.719067,2026-05-14 21:05:35+00:00
5,FRMI,25,1,0.999400,2026-05-15 15:10:18+00:00
6,AMD,6,5,0.377380,2026-05-15 15:12:08+00:00
7,INTC,4,3,0.028600,2026-05-16 20:57:23+00:00
8,ILMN,9,1,0.870400,2026-05-14 01:30:43+00:00
9,VG,9,1,0.998300,2026-05-14 11:08:46+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['LNG', 'RKLB', 'NBIS', 'APLD']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
    analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": float(features_df["Close"].dropna().iloc[-1]),
        "latest_rsi": float(features_df["RSI"].dropna().iloc[-1]),
        "latest_macd": float(features_df["MACD"].dropna().iloc[-1]),
        "return_21d": float(features_df["21D Return"].dropna().iloc[-1]),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,APLD,251,42.560001,60.845832,4.041879,0.414423,0.776241,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
1,LNG,251,241.839996,37.718043,-6.133373,-0.079546,4.034772,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
2,NBIS,251,219.940002,72.139888,19.891508,0.330229,7.715559,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
3,RKLB,251,124.769997,70.818158,12.835091,0.504522,4.433020,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== LNG latest metrics =====
                 Close   Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI      MACD  5D Return  21D Return
Date                                                                                                                                    
2026-05-11  240.699997  2117700           13.042949          265.199001           248.201549  34.481170 -3.729459  -0.117830   -0.093545
2026-05-12  244.309998  2130200           12.428805          263.554667           248.112376  37.990125 -4.445363  -0.093537   -0.066093
2026-05-13  239.380005  2120800           12.581166          262.075334           247.912294  35.216300 -5.348873  -0.084309   -0.078173
2026-05-14  241.080002  2055700           12.867159          260.916667           247.755759  36.926480 -5.860183  -0.023097   -0.061032
2026-05-15  241.839996  2682100           12.735967          259.606000           247.620232  37.718043 -6.133373   0.007205   -0.079546

===== RK

## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== LNG regression results =====
                 Close  Predicted Close
Date                                   
2026-05-04  272.850006       272.980368
2026-05-05  269.519989       269.495824
2026-05-06  261.420013       262.926289
2026-05-07  246.779999       254.769582
2026-05-08  240.110001       248.841501
2026-05-11  240.699997       244.986673
2026-05-12  244.309998       245.329254
2026-05-13  239.380005       242.633283
2026-05-14  241.080002       245.253428
2026-05-15  241.839996       245.685845

===== RKLB regression results =====
                 Close  Predicted Close
Date                                   
2026-05-04   80.309998        84.037122
2026-05-05   78.760002        83.721779
2026-05-06   84.650002        86.972348
2026-05-07   78.580002        83.382939
2026-05-08  105.470001        98.503336
2026-05-11  117.349998       106.258091
2026-05-12  117.559998       111.187518
2026-05-13  124.150002       116.262147
2026-05-14  132.550003       125.582539
2026-05

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "LNG",
    "RKLB",
    "NBIS",
    "APLD"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
